# Stage 4 — Model 3: Histogram-Based Gradient Boosting (`HistGradientBoostingClassifier`)
**Project:** 10-Year Coronary Heart Disease (CHD) Risk Prediction (Framingham Heart Study)  
**Author:** Saif Asaad Naeem | Al-Maaqal University — Department of AI Engineering

---
### Notebook Roadmap:
1. **Environment Setup & Data Loading:** Load `framingham_balanced.csv` and apply the unified 80%/20% stratified split.
2. **Stage 1 — Baseline Model Evaluation:** Evaluate the default `HistGradientBoostingClassifier`.
3. **Stage 2 — Continuous Exploration via `RandomizedSearchCV ()`:** Sample hyperparameter distributions (`randint`, `loguniform`, `uniform`) using parentheses `()`.
4. **Stage 3 — Exhaustive Exploitation via `GridSearchCV []`:** Perform localized grid refinement using square brackets `[]`.
5. **Final Results Comparison, Serialization & Diagnostic Plots:** Export `best_model3_histgb.pkl` and generate diagnostic plots.

### 1. Environment Setup and Data Loading
Import required packages and initialize the 5-Fold Stratified Cross-Validation evaluator.

In [ ]:
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint, uniform, loguniform

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, GridSearchCV, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, RocCurveDisplay

df = pd.read_csv('framingham_balanced.csv')
X = df.drop('TenYearCHD', axis=1).values
y = df['TenYearCHD'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

def evaluate_model(model, stage_name):
    cv_res = cross_validate(model, X_train, y_train, cv=skf, scoring=scoring_metrics, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return {
        'Stage': stage_name,
        'CV_Recall (%)': round(cv_res['test_recall'].mean() * 100, 2),
        'CV_Precision (%)': round(cv_res['test_precision'].mean() * 100, 2),
        'CV_F1 (%)': round(cv_res['test_f1'].mean() * 100, 2),
        'CV_ROC_AUC (%)': round(cv_res['test_roc_auc'].mean() * 100, 2),
        'CV_Accuracy (%)': round(cv_res['test_accuracy'].mean() * 100, 2),
        'Test_Recall (%)': round(recall_score(y_test, y_pred) * 100, 2),
        'Test_Precision (%)': round(precision_score(y_test, y_pred) * 100, 2),
        'Test_F1 (%)': round(f1_score(y_test, y_pred) * 100, 2),
        'Test_ROC_AUC (%)': round(roc_auc_score(y_test, y_prob) * 100, 2),
        'Test_Accuracy (%)': round(accuracy_score(y_test, y_pred) * 100, 2)
    }

print(f"Dataset Loaded: Train = {X_train.shape}, Test = {X_test.shape}")

### 2. Stage 1: Baseline `HistGradientBoostingClassifier` Evaluation
Evaluate the untuned baseline histogram-based gradient boosting classifier.

In [ ]:
base_hgb = HistGradientBoostingClassifier(random_state=42)
res_base = evaluate_model(base_hgb, '1. Baseline HistGradientBoosting')
print("Baseline Evaluation:", res_base)

### 3. Stage 2: Broad Exploration via `RandomizedSearchCV ()`
Sample continuous and discrete probability distributions enclosed in parentheses `()`.

In [ ]:
param_dist_hgb = {
    'max_iter': randint(150, 450),
    'learning_rate': loguniform(0.02, 0.25),
    'max_leaf_nodes': randint(31, 95),
    'max_depth': randint(8, 25),
    'min_samples_leaf': randint(5, 25),
    'l2_regularization': uniform(0.0, 1.5)
}

rand_hgb = RandomizedSearchCV(
    estimator=HistGradientBoostingClassifier(random_state=42),
    param_distributions=param_dist_hgb,
    n_iter=30,
    scoring='f1',
    cv=skf,
    random_state=42,
    n_jobs=-1
)
rand_hgb.fit(X_train, y_train)
best_r = rand_hgb.best_params_
print("Best Parameters from RandomizedSearchCV ():", best_r)
res_rand = evaluate_model(rand_hgb.best_estimator_, '2. RandomizedSearchCV ()')

### 4. Stage 3: Fine-Grained Exploitation via `GridSearchCV []`
Refine the optimal neighborhood using an explicit parameter grid inside square brackets `[]`.

In [ ]:
param_grid_hgb = {
    'max_iter': [max(100, best_r['max_iter'] - 40), best_r['max_iter'], best_r['max_iter'] + 40],
    'learning_rate': [round(max(0.01, best_r['learning_rate'] * 0.8), 4), round(best_r['learning_rate'], 4), round(min(0.4, best_r['learning_rate'] * 1.2), 4)],
    'max_leaf_nodes': [max(15, best_r['max_leaf_nodes'] - 10), best_r['max_leaf_nodes'], best_r['max_leaf_nodes'] + 10],
    'max_depth': [max(5, best_r['max_depth'] - 2), best_r['max_depth'], best_r['max_depth'] + 2],
    'min_samples_leaf': [max(2, best_r['min_samples_leaf'] - 3), best_r['min_samples_leaf'], best_r['min_samples_leaf'] + 3],
    'l2_regularization': [round(best_r['l2_regularization'], 3)]
}

grid_hgb = GridSearchCV(
    estimator=HistGradientBoostingClassifier(random_state=42),
    param_grid=param_grid_hgb,
    scoring='f1',
    cv=skf,
    n_jobs=-1
)
grid_hgb.fit(X_train, y_train)
print("Best Parameters from GridSearchCV []:", grid_hgb.best_params_)
res_grid = evaluate_model(grid_hgb.best_estimator_, '3. GridSearchCV [] (Final)')

### 5. Final Results Comparison, Model Serialization & Diagnostic Plots
Summarize optimization stages, export `best_model3_histgb.pkl`, and plot the Confusion Matrix and ROC curve.

In [ ]:
results_df = pd.DataFrame([res_base, res_rand, res_grid])
print("\n=== Model 3 (HistGradientBoostingClassifier) Optimization Progression Table ===")
print(results_df.to_string(index=False))

joblib.dump(grid_hgb.best_estimator_, 'best_model3_histgb.pkl')
print("\nSUMMARY_MODEL_3_HISTGB:", {'best_params': grid_hgb.best_params_, 'metrics': res_grid})

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
y_pred_final = grid_hgb.best_estimator_.predict(X_test)
sns.heatmap(confusion_matrix(y_test, y_pred_final), annot=True, fmt='d', cmap='Oranges', ax=axes[0])
axes[0].set_title('HistGradientBoosting Confusion Matrix (Test Set)', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
RocCurveDisplay.from_estimator(grid_hgb.best_estimator_, X_test, y_test, ax=axes[1], name='HistGB (GridSearch)')
axes[1].set_title('HistGradientBoosting ROC Curve', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_model3_histgb.png', dpi=300)
plt.show()